# 05 – SVM tuyến tính: ablation đặc trưng, tinh chỉnh C, kiểm chứng Group split

**Người thực hiện:** danh (Huỳnh Quốc Danh)  
**Mục tiêu:** Trả lời RQ1 (đặc trưng nào giúp) và chọn mô hình cuối cùng cho hệ thống.

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> danh`.

**Các bước:**
1. Ablation: word → char → hybrid → full (cùng C).
2. Thử char_ngram (2,4), (3,5), (3,6).
3. Quét C ∈ {0.03, 0.1, 0.3, 1, 3} có CV.
4. class_weight='balanced' có giúp 3 lớp tủ lạnh/tủ đông?
5. Chạy lại cấu hình tốt nhất với split='group'.
6. GridSearchCV nhỏ (tùy chọn) và kết luận mô hình chọn.

In [1]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


## 1. Ablation đặc trưng

In [2]:
res, _ = run(member='danh', model='svc', features='word', params={'C': 1.0}, exp_name='abl_word', show_report=False)

[danh_20261113_164215] svc/word {'C': 1.0} split=strat
   train_time_s                 1.7574
   test_accuracy                0.9669
   test_precision_macro         0.9685
   test_recall_macro            0.9674
   test_f1_macro                0.9679
   test_f1_weighted             0.9669
   f1_Fridges                   0.9234
   f1_Freezers                  0.9530
   f1_Fridge_Freezers           0.9495
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính:SVM (LinearSVC, C=1) chỉ dùng word TF-IDF (1,2): Macro-F1 0.9679, Accuracy 0.9669; F1 Fridges 0.9234, Freezers 0.9530, Fridge Freezers 0.9495 (run_id danh_20261113_164215). Chạy lại ra đúng bằng ngày 12/11 (danh_20261112_124307) → kết quả tái lập được.
> - Nhận xét: đây là mốc so sánh. Ba lớp tủ lạnh/tủ đông yếu nhất (đều dưới 0.96), Fridges thấp nhất, vì nhiều tiêu đề chỉ có hãng + mã model mà word n-gram không đọc được. Thêm char n-gram (hybrid 2,4) tăng lên 0.9924 (+2,45 điểm); thêm hình thái + merchant (full) chỉ được 0.9920, không hơn hybrid.
> - Lần sau thử: dùng hybrid làm đặc trưng chính, so phạm vi char (2,4)/(3,5)/(2,5)/(3,6) bằng 5-fold CV.

Đổi `features` thành `'char'`, `'hybrid'`, `'full'` và chạy lại – mỗi lần ghi nhận xét.

## 2. Phạm vi n-gram ký tự

In [3]:
res, _ = run(member='danh', model='svc', features='hybrid', params={'C': 1.0}, char_ngram=(2, 4), exp_name='char_24', show_report=False)

[danh_20261113_164224] svc/hybrid {'C': 1.0} split=strat
   train_time_s                 6.5794
   test_accuracy                0.9929
   test_precision_macro         0.9927
   test_recall_macro            0.9921
   test_f1_macro                0.9924
   test_f1_weighted             0.9929
   f1_Fridges                   0.9818
   f1_Freezers                  0.9739
   f1_Fridge_Freezers           0.9918
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: SVM (C=1) với hybrid TF-IDF = word (1,2) + char_wb (2,4): Macro-F1 0.9924, Accuracy 0.9929; F1 Fridges 0.9818, Freezers 0.9739, Fridge Freezers 0.9918; train 6.6 s.
> - Nhận xét:  so với chỉ dùng word (0.9679), thêm n-gram ký tự tăng +2,45 điểm; Fridges tăng mạnh nhất (0.923 → 0.982). Lý do: char n-gram đọc được tiền tố mã model (vd "kif", "ks") và từ tiếng Đức mất dấu mà word n-gram bỏ sót. Các phạm vi khác: (2,5) 0.9924 nhưng chậm hơn, (3,5) 0.9917, (3,6) 0.9910. CV xác nhận (2,4) = 0.99066 ± 0.00095 cao hơn (3,5) = 0.98890 ± 0.00080, chênh lớn hơn std nên tốt hơn thật.
> - Lần sau thử: giữ char (2,4), quét C = 1, 3, 10 có CV.

## 3. Quét C (có 5-fold CV)

In [4]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 0.3}, cv=True, exp_name='C_0.3', show_report=False)

[danh_20261113_164354] svc/full {'C': 0.3} split=strat
   cv_f1_macro_mean             0.9866
   cv_f1_macro_std              0.0010
   cv_time_s                    61.8752
   train_time_s                 14.1713
   test_accuracy                0.9904
   test_precision_macro         0.9905
   test_recall_macro            0.9895
   test_f1_macro                0.9900
   test_f1_weighted             0.9904
   f1_Fridges                   0.9709
   f1_Freezers                  0.9748
   f1_Fridge_Freezers           0.9891
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 4. class_weight

In [5]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 1.0, 'class_weight': 'balanced'}, exp_name='balanced')

[danh_20261113_164414] svc/full {'C': 1.0, 'class_weight': 'balanced'} split=strat
   train_time_s                 16.5774
   test_accuracy                0.9921
   test_precision_macro         0.9917
   test_recall_macro            0.9912
   test_f1_macro                0.9914
   test_f1_weighted             0.9921
   f1_Fridges                   0.9791
   f1_Freezers                  0.9727
   f1_Fridge_Freezers           0.9918
                  precision    recall  f1-score   support

            CPUs     0.9987    0.9987    0.9987       773
 Digital Cameras     0.9944    0.9944    0.9944       540
     Dishwashers     0.9913    0.9942    0.9927       685
        Freezers     0.9772    0.9683    0.9727       442
 Fridge Freezers     0.9918    0.9918    0.9918      1100
         Fridges     0.9804    0.9777    0.9791       717
      Microwaves     0.9979    0.9979    0.9979       468
   Mobile Phones     0.9951    1.0000    0.9976       816
             TVs     0.9972    0.9944    0

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 5. Group split theo Cluster ID
Câu hỏi phải trả lời được: *vì sao F1 giảm khi chia theo sản phẩm? giảm bao nhiêu là chấp nhận được?*

In [6]:
res, _ = run(member='danh', model='svc', features='full', params={'C': 1.0}, split='group', exp_name='group_best', show_report=False)

[danh_20261113_164436] svc/full {'C': 1.0} split=group
   train_time_s                 17.4306
   test_accuracy                0.9871
   test_precision_macro         0.9877
   test_recall_macro            0.9860
   test_f1_macro                0.9868
   test_f1_weighted             0.9871
   f1_Fridges                   0.9614
   f1_Freezers                  0.9761
   f1_Fridge_Freezers           0.9804
→ Đã ghi vào experiments/danh/experiment_log.csv. Hãy mở file và điền cột nhan_xet.


> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/danh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 6. Bảng so sánh & hình

In [7]:
log = pd.read_csv('../experiments/danh/experiment_log.csv')
log[['run_id','ten_thi_nghiem','features','params','split','test_f1_macro','f1_Fridges','f1_Freezers','nhan_xet']]

ParserError: Error tokenizing data. C error: Expected 26 fields in line 7, saw 51


## 7. Mô hình chọn và lý do (tự viết – đây là câu thầy sẽ hỏi)
- Đặc trưng: … vì …
- Mô hình: … vì …
- Tham số: … (dẫn run_id chứng minh)